# SCENE `base_schemas`: what is new, in one demo

Registers one Mathis-lab mouse and one session on the shared SCENE tables, and shows what changed since `main`.

Needs a MySQL database (`make db_up`). Schemas are created under the prefix `demo_`.

| New | Where | In short |
|---|---|---|
| Lazy schemas + registry | `core.SCENE_REGISTRY` | import without a database, activate when ready |
| Shared tables | `schemas.scene` | `Lab`, `Project`, `Subject`, `Session` instead of lab-specific `mice` / `exp` |
| Write helpers | `ingestion` | mint ids, validate pseudonymous codes, write atomically |
| `DuplicatePolicy` | `ingestion.provenance` | what happens when an entry already exists |
| Row stamps | `schemas.provenance` | per row: deployment, writer version, content hash |
| Schema versions | `SchemaVersion` | recorded on creation, checked before the first write |
| Table markers | `core` | `SyncAuthority`, `WriteRole`: intent for sync and database grants |

In [13]:
import os
import sys
import warnings
from datetime import date
from pathlib import Path

warnings.filterwarnings("ignore", message="No datajoint.json found")
import datajoint as dj

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "base_schemas").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "examples")]

# Connection: shell variables win, then the repo .env. The schema prefix is not
# taken from .env, so the demo never writes into dev_ / production schemas.
prefix = os.environ.get("DJ_SCHEMA_PREFIX", "demo_")
env_file = ROOT / ".env"
for line in env_file.read_text().splitlines() if env_file.is_file() else []:
    key, sep, value = line.partition("=")
    if sep and not key.strip().startswith("#"):
        os.environ.setdefault(key.strip(), value.strip().strip("'\""))
os.environ["DJ_SCHEMA_PREFIX"] = prefix
os.environ["AUTO_ACTIVATE"] = ""  # keep schemas unbound on import (section 1)

# Identity of *this* database, stamped on every row written below (section 6).
os.environ["SCENE_DEPLOYMENT_ID"] = "mlai-demo"
os.environ["SCENE_DEPLOYMENT_LABEL"] = "Mathis lab, demo database"

dj.config["database.host"] = os.environ.get("DJ_HOST", "127.0.0.1")
dj.config["database.port"] = int(
    os.environ.get("DJ_PORT") or os.environ.get("MYSQL_PUBLISH_PORT", 3306)
)
dj.config["database.user"] = os.environ.get("DJ_USER", "root")
dj.config["database.password"] = os.environ.get("DJ_PASS") or os.environ.get(
    "MYSQL_ROOT_PASSWORD", ""
)

## 1. Lazy schemas and the registry

Importing the tables needs no database. Each schema registers itself in `SCENE_REGISTRY` and stays unbound until it is activated. A lab registers its own schema the same way (section 5).

In [ ]:
from base_schemas.core import SCENE_REGISTRY
from base_schemas.schemas.provenance.deployment import Deployment
from base_schemas.schemas.provenance.row_meta import SessionRowMeta, SubjectRowMeta
from base_schemas.schemas.scene.lab import Lab
from base_schemas.schemas.scene.project import Project
from base_schemas.schemas.scene.session import Session
from base_schemas.schemas.scene.subject import Subject

In [3]:
# Activate all SCENE schemas in the registry, and print their database name
SCENE_REGISTRY.activate_all()

for _, schema in SCENE_REGISTRY.schemas.items():
    print(schema.database)

## 2. Table markers

Every table says which database holds the truth (`SyncAuthority`) and who may insert (`WriteRole`). They record intent only: database grants and sync tooling act on them.

In [5]:
from base_schemas.core import sync_authority_of, write_role_of

for table in (Lab, Project, Subject, Session, Session.Subject, SessionRowMeta, Deployment):
    role = write_role_of(table)
    print(
        f"{table.__qualname__:16} truth: {sync_authority_of(table).value:8} insert: {role.value if role else 'anyone'}"
    )

Lab              truth: central  insert: admin
Project          truth: central  insert: admin
Subject          truth: origin   insert: acquisition
Session          truth: origin   insert: acquisition
Session.Subject  truth: origin   insert: acquisition
SessionRowMeta   truth: origin   insert: acquisition
Deployment       truth: shared   insert: anyone
Mouse            truth: origin   insert: anyone


## 3. Register the lab's data on the shared tables

**Admin:** Write `Lab` and `Project` using the admin the helpers. These are consortium-wide catalogs.

In [8]:
from base_schemas.ingestion.scene import admin
from base_schemas.ingestion.provenance import DuplicatePolicy

lab = admin.ensure_lab(
    {"lab_id": "mlai", "lab_name": "Mathis Lab of Adaptive Intelligence", "institution": "EPFL"},
    if_exists=DuplicatePolicy.VERIFY,
)
project = admin.ensure_project(
    {
        "project_name": "mouse_ar_hockey",
        "project_title": "The augmented reality hockey project of the mouse team.",
    },
    if_exists=DuplicatePolicy.VERIFY,
)
Lab()

lab_id,lab_name,institution
mlai,Mathis Lab of Adaptive Intelligence,EPFL


**Acquisition:** Subjects and sessions are everyday writes. The lab picks a pseudonymous code (unique WITHIN the lab); the helper mints the globally unique id the first time and returns the same id afterwards. `atomic` makes several registrations all-or-nothing.

In [10]:
from base_schemas.core import atomic
from base_schemas.ingestion import register_session, register_subject

with atomic(Session.connection):
    subject = register_subject("mouse-042", "mouse", lab=lab)
    session = register_session(
        "ar-2026-05-01-1", date(2026, 5, 1), lab=lab, subjects=[subject], project=project
    )

print(subject, session, sep="\n")

{'subject_id': '9ae1e95a87934d9f8db5afd966481251'}
{'lab_id': 'mlai', 'session_id': '8f9cb798bee34e8ba535f3c48b93926a'}


## 4. Registering again: `DuplicatePolicy`

`REJECT` raises, `SKIP` keeps the stored row, `VERIFY` accepts identical content and raises on a difference, `OVERWRITE` rewrites the row in place (omitted fields go back to their default).

In [11]:
changed = {"project_name": "mouse_ar_hockey", "project_title": "A different title"}
original = {
    "project_name": "mouse_ar_hockey",
    "project_title": "The augmented reality hockey project of the mouse team.",
}

print("VERIFY, same content ->", admin.ensure_project(original, if_exists=DuplicatePolicy.VERIFY))
for policy in (DuplicatePolicy.REJECT, DuplicatePolicy.VERIFY):
    try:
        admin.ensure_project(changed, if_exists=policy)
    except ValueError as err:
        print(f"{policy.name:9} ->", err)

admin.ensure_project(changed, if_exists=DuplicatePolicy.SKIP)
print("SKIP      -> stored title:", (Project & project).fetch1("project_title"))

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    admin.ensure_project(changed, if_exists=DuplicatePolicy.OVERWRITE)
    print(
        "OVERWRITE -> stored title:",
        (Project & project).fetch1("project_title"),
        "| warning:",
        caught[-1].message,
    )
    admin.ensure_project(original, if_exists=DuplicatePolicy.OVERWRITE)  # put it back

VERIFY, same content -> {'project_name': 'mouse_ar_hockey'}
REJECT    -> Project row {'project_name': 'mouse_ar_hockey'} is already registered
VERIFY    -> Project row {'project_name': 'mouse_ar_hockey'} is already registered with a different content hash
SKIP      -> stored title: The augmented reality hockey project of the mouse team.
OVERWRITE -> stored title: A different title | warning: Project row {'project_name': 'mouse_ar_hockey'} content hash changed; overwriting


## 5. Link the lab's existing tables to the shared rows

The lab already has its own pipeline (`examples/mlai_schemas/mice.py`, `exp.py`) with its own keys. It stays as it is: the mouse and the session are inserted the way the lab always did.

In [ ]:
from mlai_schemas.exp import Session as LabSession
from mlai_schemas.mice import Mouse

Mouse.insert1(
    {"mouse_name": "Axolotl", "mouse_id": 42, "dob": "2025-10-01", "sex": "M", "strain": "N/A"},
    skip_duplicates=True,
)
LabSession.insert1(
    {
        "mouse_name": "Axolotl",
        "day": 1,
        "attempt": 1,
        "doe": "2026-05-01",
        "session_increment": 1,
        "rig_id": 1,
        "experimenter_name": "user",
        "anesthesia_name": "awake",
        "opto_name": "none",
        "task_name": "AR_visual_discrimination",
        "session_notes": "first AR session",
    },
    skip_duplicates=True,
)
LabSession()

Two small mapping tables connect those lab rows to the shared `Subject` and `Session`. They live in the lab's own schema, registered on the same registry.

In [ ]:
mlai = SCENE_REGISTRY.make_schema("mlai")


@mlai
class MouseMapping(dj.Manual):
    definition = """
    -> Subject
    ---
    -> Mouse
    """


@mlai
class SessionMapping(dj.Manual):
    definition = """
    -> Session
    ---
    -> LabSession
    """


SCENE_REGISTRY.activate("mlai")

MouseMapping.insert1({**subject, "mouse_name": "Axolotl"}, skip_duplicates=True)
SessionMapping.insert1(
    {**session, "mouse_name": "Axolotl", "day": 1, "attempt": 1}, skip_duplicates=True
)

display(Subject * MouseMapping * Mouse)
Session.proj("session_code") * SessionMapping * LabSession.proj("doe", "rig_id", "task_name")

### Extending the shared layer with a new lab table

New lab-only information hangs directly off the shared row: no second identity for the same session.

In [ ]:
@mlai
class SessionContext(dj.Manual):
    definition = """
    -> Session
    ---
    room: varchar(64)
    notes='': varchar(255)
    """


SessionContext.insert1({**session, "room": "B1.05", "notes": "new AR rig"}, skip_duplicates=True)
Session.proj("session_code", "session_date") * SessionContext

In [ ]:
(
    dj.Diagram(SCENE_REGISTRY.get("scene"))
    + dj.Diagram(mlai)
    + dj.Diagram(Mouse)
    + dj.Diagram(LabSession)
)

## 6. What syncing between databases will use

Each database has an identity (`Deployment`), set once with `SCENE_DEPLOYMENT_ID`.

In [11]:
Deployment()

deployment_id,label
mlai-demo,"Mathis lab, demo database"


Every helper also wrote a **stamp** next to the row: who wrote it (`deployment_id`), with which writer (`ingestion_version`), and a hash of the content. Two databases hold the same entry when key and `content_hash` agree; a different hash means the content differs.

In [12]:
display(Subject.proj("subject_code") * SubjectRowMeta)
Session.proj("session_code") * SessionRowMeta

subject_id,subject_code,deployment_id,ingestion_version,content_hash,updated_at
a48230ce575a463aab6482a4e1bd954d,mouse-042,mlai-demo,0.0.1,9450e51dbfea0008c5c7b1673528b74ef2a719506b6b54f49b68eb6f73b91c89,2026-10-02 14:17:51


lab_id,session_id,session_code,deployment_id,ingestion_version,content_hash,updated_at
mlai,e6b5c1108d564ff9b3aaadb6b5bb2d78,ar-2026-05-01-1,mlai-demo,0.0.1,797902fb948c79643d6087dc9ae93310b7e7a4ac8a5cabdc40da97db27dd63f3,2026-10-02 14:17:51


Each schema records its **definition version** when it is created. The write helpers check it against the installed code before their first write, so old code never writes into a newer database (or the reverse).

In [13]:
from base_schemas.core import assert_schema_compatible
from base_schemas.schemas.scene._schema import SCENE_SCHEMA_VERSION, SchemaVersion

print(
    "code:",
    SCENE_SCHEMA_VERSION,
    "| database:",
    assert_schema_compatible(SCENE_SCHEMA_VERSION, SchemaVersion),
)
SchemaVersion()

code: 0.0.2 | database: 0.0.2


version,applied_at,notes
0.0.2,2026-10-02 14:17:48,recorded on creation
